# 4. 多层感知机 (MLP) (下)

## 4.10 实战 Kaggle 房价预测

> 本章算是练习章节，基本不会有新知识，仅是老知识的巩固。

### **任务背景与数据流**
*   **目标**：给定房屋的 79 个特征（如面积、地段、房龄），预测房价（连续值）。
*   **评价指标**：由于房价跨度大，通常使用**对数均方根误差 (Log RMSE)**。
    *   公式：$\sqrt{\frac{1}{n} \sum_{i=1}^n (\log y_i - \log \hat{y}_i)^2}$
    *   **原因**：这样误差 10% 对豪宅和普通公寓的惩罚权重是一样的。

### 1. 数据自动化下载与管理

> 训练数据集包括1460个样本，每个样本80个特征和1个标签;
>
> 测试数据集包含1459个样本，每个样本80个特征。

In [11]:
from typing import Final
from pathlib import Path
import hashlib
import requests

# 数据源配置（D2L 官方提供的镜像，无需 Kaggle 令牌即可下载）
DATA_HUB: dict[str, tuple[str, str]] = {
    'kaggle_house_train': ('http://d2l-data.s3-accelerate.amazonaws.com/kaggle_house_pred_train.csv',
                           '585e9cc93e70b39160e7921475f9bcd7d31219ce'),  
    'kaggle_house_test': ('http://d2l-data.s3-accelerate.amazonaws.com/kaggle_house_pred_test.csv',
                          'fa19780a7b011d9b009e8bff8e99922a8ee2eb90'),
}

# 下载本地路径配置
BASE_DIR: Final[Path] = Path.cwd()
DATA_ROOT: Final[Path] = BASE_DIR / "temp" / "data"

def download_data(name: str, cache_dir: Path = DATA_ROOT / "kaggle_house") -> str:
    """下载数据并验证 SHA-1 哈希值，确保数据未损坏。
    
    Args:
        name: 数据集名称。
        cache_dir: 本地缓存目录。

    Returns:
        下载后的本地文件路径。
    """
    assert name in DATA_HUB, f"{name} 不在数据源配置中。"
    url, sha1_hash = DATA_HUB[name]
    
    # 确保 cache_dir 目录及其所有必要的上级目录被创建，且如果目录已存在，则什么也不做。
    cache_dir.mkdir(parents=True, exist_ok=True)
    file_name = cache_dir / url.split('/')[-1]

    if file_name.exists():
        sha1 = hashlib.sha1()
        with file_name.open('rb') as f:
            while True:
                # 按块读取，防止过大导致内存溢出
                data = f.read(1048576)
                if not data:
                    break
                sha1.update(data)
        if sha1.hexdigest() == sha1_hash:
            return str(file_name) # 哈希校验通过，直接返回路径

    print(f"正在从 {url} 下载 {file_name}...")
    r = requests.get(url, stream=True, verify=True)
    with file_name.open('wb') as f:
        f.write(r.content)
    return str(file_name)

# 执行下载
train_file = download_data('kaggle_house_train')
test_file = download_data('kaggle_house_test')

### 2. 实验配置

In [12]:
from dataclasses import dataclass, field
import torch

@dataclass(frozen=True)
class HouseConfig:
    """房价预测超参数配置。"""
    # K-折交叉验证超参数
    # 训练数据集平均分为 K 份，在训练时，依次选择其中 1 份作为验证集，其余 K-1 份作为训练集。
    k_folds: int = 5                

    num_epochs: int = 100
    lr: float = 5.0             # Adam 优化器对大步长较稳健
    batch_size: int = 64
    weight_decay: float = 0.0   # 调参初期通常不加惩罚项，仅在观察到明显过拟合时，才引入惩罚项。
    device: str = "cude" if torch.cuda.is_available() else "cpu"

@dataclass(frozen=True)
class MLPConfig(HouseConfig):
    """多层感知机进阶配置。"""
    # 仅覆盖需要修改的默认参数
    lr: float = 0.01            # MLP 通常需要比线性模型更小的学习率
    weight_decay: float = 0.1   # 显式开启 L2 正则化 (深层神经网络更容易过拟合)
    
    # MLP 独有参数
    num_hiddens: list[int] = field(default_factory=lambda: [256, 128])
    dropout_prob: list[float] = field(default_factory=lambda: [0.2, 0.5])

### 3. 数据预处理 (特征工程)

> 表格数据的核心在于如何处理**缺失值**和**非数值类型**。

In [38]:
import pandas as pd
from torch import Tensor
def preprocess_features(train_df: pd.DataFrame, test_df: pd.DataFrame) -> tuple[Tensor, Tensor, Tensor]:
    """数据标准化与编码。
    
    处理流程：
    1. 特征对齐：合并训练集与测试集特征，确保独热编码后的维度一致。
    2. 数值标准化：将数值特征缩放到均值 0，方差 1，并填充缺失值为 0。
    3. 类别编码：执行独热编码（One-hot Encoding），将缺失类别显式化。
    4. 格式转换：将 Pandas DataFrame 转换为 PyTorch Float32 Tensor。

    Args:
        train_df: 原始训练数据，包含 Id 和 SalePrice。
        test_df: 原始测试数据，包含 Id，不含 SalePrice。

    Returns:
        tuple[Tensor, Tensor, Tensor]: (训练特征, 测试特征, 训练标签)
    """
    # 0. 合并特征以便统一处理编码 (去掉 Id 和 Label)
    # iloc：Pandas 中用于 基于整数位置（Integer-based Indexing） 检索数据的方法。
    all_features = pd.concat((train_df.iloc[:, 1:-1], test_df.iloc[:, 1:]))

    # 1. 处理数值特征。
    # 选择数字类型的列。
    # 计算每个特征的均值和标准差，应用 (x - mean) / std。
    # 填充数值缺失值。标准化后，特征均值为 0，因此用 0 填充缺失值在统计学上是合理的（代表平均水平）。
    numeric_features = all_features.select_dtypes(include=['number']).columns
    all_features[numeric_features] = all_features[numeric_features].apply(
        lambda x: (x - x.mean()) / (x.std())
    )
    all_features[numeric_features] = all_features[numeric_features].fillna(0)

    # 2. 处理类别特征 (One-hot)。
    # get_dummies 将类别转换为多列 0/1。
    # dummy_na = True 会为缺失值 (NaN) 自动创建一列
    all_features = pd.get_dummies(all_features, dummy_na=True, dtype=float)

    # 3. 数据切分与格式转换。
    n_train = train_df.shape[0]
    train_features = torch.tensor(all_features[:n_train].values, dtype=torch.float32)
    test_features = torch.tensor(all_features[n_train:].values, dtype=torch.float32)
    train_labels = torch.tensor(train_df.SalePrice.values.reshape(-1, 1), dtype=torch.float32)

    return train_features, test_features, train_labels


### 4. 模型架构与评估函数

> 房价差异巨大（10万 vs 100万），我们关心的是**相对误差**。

公式：$\sqrt{\frac{1}{n} \sum_{i=1}^n (\log y_i - \log \hat{y}_i)^2}$

In [39]:
from torch import nn
def get_model(num_inputs: int, num_outputs: int) -> nn.Sequential:
    """构建线性回归模型。"""
    return nn.Sequential(nn.Linear(num_inputs, num_outputs))

def log_rmse(net: nn.Module, features: Tensor, labels: Tensor) -> float:
    """计算对数均方根误差 (Log-RMSE)。"""
    # 限制预测值在 1 以上，防止 log(0) 报错。
    # 下限：1; 上限：'inf'(即 无)。
    clipped_preds = torch.clamp(net(features), 1, float('inf'))
    rmse = torch.sqrt(nn.MSELoss()(torch.log(clipped_preds), torch.log(labels)))
    return rmse.item()

In [40]:
# 进阶实现
def get_mlp_net(num_inputs: int, num_outputs:int, config: MLPConfig) -> nn.Sequential:
    """构建多层感知机 (MLP) 模型。
    
    结构：线性 -> ReLU -> Dropout -> 线性 -> ReLU -> Dropout -> 输出。

    Args:
        num_inputs: 输入特征的维度。
        num_outputs: 输出标签的维度。
        config: 包含隐藏层维度和丢弃率的配置对象。

    Returns:
        nn.Sequential: 组装好的深度学习模型。
    """
    # nn.Flatten(): 确保传入的数据一定是符合全连接层要求的平面结构。
    layers: list[nn.Module] = [nn.Flatten()]

    curr_dim = num_inputs
    # 动态构建隐藏层
    for h_dim, dropout_prob in zip(config.num_hiddens, config.dropout_prob):
        layers.append(nn.Linear(curr_dim, h_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout_prob))
        curr_dim = h_dim

    # 输出层
    layers.append(nn.Linear(curr_dim, num_outputs))

    net = nn.Sequential(*layers)

    # 应用 Kaiming 初始化 (针对 ReLU 优化)
    def init_weights(m: nn.Module) -> None:
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
    
    net.apply(init_weights)
    return net

### 5. 训练引擎与 K 折验证

In [41]:
from typing import Optional
from torch.utils.data import DataLoader, TensorDataset
def train_model(
    net: nn.Module, train_features: Tensor, train_labels: Tensor,
    valid_features: Optional[Tensor], valid_labels: Optional[Tensor],
    config: HouseConfig
) -> tuple[list[float], list[float]]:
    """使用 Adam 优化器训练模型并记录训练与验证集的 Log-RMSE。
    
    Args:
        net: 待训练的 PyTorch 模型。
        train_features: 训练集特征张量，形状为 (样本数, 特征数)。
        train_labels: 训练集标签张量，形状为 (样本数, 1)。
        valid_features: 可选的验证集特征张量。
        valid_labels: 可选的验证集标签张量。
        config: 包含 batch_size, lr, weight_decay, num_epochs 等超参数的配置对象。

    Returns:
        tuple: 包含两个列表 (train_ls, valid_ls)，分别记录每个 epoch 的对数均方根误差。
    """
    train_ls, valid_ls = [], []
    train_iter = DataLoader(TensorDataset(train_features, train_labels), config.batch_size, shuffle=True)

    # 使用 Adam 优化器，对初始学习率不敏感
    optimizer = torch.optim.Adam(net.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    # 定义损失函数
    loss_fn = nn.MSELoss()

    for epoch in range(config.num_epochs):
        net.train()
        for X, y in train_iter:
            optimizer.zero_grad()
            l = loss_fn(net(X), y)
            l.backward()
            optimizer.step()
        
        # 评估阶段
        net.eval()
        with torch.no_grad():
            train_ls.append(log_rmse(net, train_features, train_labels))
            if valid_labels is not None:
                valid_ls.append(log_rmse(net, valid_features, valid_labels))

    return train_ls, valid_ls

def get_k_fold_data(k_folds: int, valid_fold_idx: int, features: Tensor, labels: Tensor) -> tuple[Tensor, Tensor, Tensor, Tensor]:
    """将原始数据集划分为 K 个子集，提取指定索引作为验证集，其余拼接为训练集。
    
    Args:
        k_folds: 交叉验证的总折数。
        valid_fold_idx: 当前作为验证集的折索引（范围从 0 到 k_folds - 1）。
        features: 总特征张量，形状为 (样本总数, 特征维度)。
        labels: 总标签张量，形状为 (样本总数, 1)。

    Returns:
        tuple: 包含以下四个 Tensor 的元组:
            - train_features: 训练集特征。
            - train_labels: 训练集标签。
            - valid_features: 验证集特征。
            - valid_labels: 验证集标签。
    """
    # 计算每一折的基础标签数量
    fold_size = features.shape[0] // k_folds
    
    train_features, train_labels = None, None
    valid_features, valid_labels = None, None

    for fold_idx in range(k_folds):
        # 计算当前折的数据切片范围
        # 计算起始位置
        start = fold_idx * fold_size
        # 如果是最后一折，直接取到末尾，防止整除不尽导致丢失数据
        end = (fold_idx + 1) * fold_size if fold_idx != k_folds - 1 else features.shape[0]
        fold_slice = slice(start, end)
        curr_features = features[fold_slice, :]
        curr_labels = labels[fold_slice]

        if fold_idx == valid_fold_idx:
            # 当前折为验证集
            valid_features, valid_labels = curr_features, curr_labels
        elif train_features is None:
            # 遇到第一个非验证折，作为训练集的初始化块
            train_features, train_labels = curr_features, curr_labels
        else:
            # # 将后续的非验证折在行 (dim=0) 上拼接
            train_features = torch.cat([train_features, curr_features], dim=0)
            train_labels = torch.cat([train_labels, curr_labels], dim=0)

    return train_features, train_labels, valid_features, valid_labels


### 6. 全流程执行

In [42]:
# 辅助对比实验函数
def k_fold_experiment(config: HouseConfig, model_type: str = "linear") -> float:
    """K 折交叉检验。
    
    Args: 
        config: 实验超参数配置。
        model_type: 模型类型，可选 "linear" 或 "mlp"。
    """
    # 1. 读取原始数据
    train_file = download_data('kaggle_house_train')
    test_file = download_data('kaggle_house_test')
    train_data = pd.read_csv(train_file)
    test_data = pd.read_csv(test_file)

    # 2. 预处理
    train_features, test_features, train_labels = preprocess_features(train_data, test_data)
    num_inputs, num_outputs = train_features.shape[1], 1

    # 3. K 折交叉检验
    print(f"\n>>> 开始评估[{model_type.upper()}] 模型...")
    valid_l_sum = 0

    for i in range(config.k_folds):
        data = get_k_fold_data(config.k_folds, i, train_features, train_labels)

        # 根据类型初始化模型
        if model_type == "mlp" and isinstance(config, MLPConfig):
            net = get_mlp_net(num_inputs, num_outputs, config)
        else:
            net = get_model(num_inputs, num_outputs)
        
        train_ls, valid_ls = train_model(net, *data, config)
        valid_l_sum += valid_ls[-1]
        print(f"折 {i+1} | 训练 Log-RMSE: {train_ls[-1]:.4f} | 验证 Log-RMSE: {valid_ls[-1]:.4f}")

    avg_score = valid_l_sum / config.k_folds
    print(f"\n平均验证 Log-RMSE: {avg_score:.4f}")
    return avg_score

In [ ]:
# 1. 参数配置
# 方案 A. 基础线性模型
config_linear = HouseConfig()
config_mlp = MLPConfig()

# 2. 开始对比实验
score_linear = k_fold_experiment(config_linear, "linear")
score_mlp = k_fold_experiment(config_mlp, "mlp")

# 3. 结果汇总
print("\n" + "=" * 30)
print(f"实验结论: ")
print(f"Linear 模型平均误差: {score_linear:.4f}")
print(f"MLP 模型平均误差:    {score_mlp:.4f}")

# 4. 根据实验结果进行最终预测并生成提交文件
best_type = "mlp" if score_mlp < score_linear else "linear"
best_config = config_mlp if best_type == "mlp" else config_linear

print(f"\n>>> 自动选择最优模型 [{best_type.upper()}] 生成提交文件...")

# 重新准备全量数据 (不带切分)
train_file = download_data('kaggle_house_train')
test_file = download_data('kaggle_house_test')
train_data = pd.read_csv(train_file)
test_data = pd.read_csv(test_file)
train_features, test_features, train_labels = preprocess_features(train_data, test_data)

# 根据胜出者创建模型
if best_type == "mlp":
    net = get_mlp_net(train_features.shape[1], 1, best_config)
else:
    net = get_model(train_features.shape[1], 1)

# 使用全量训练集进行最后的训练 (无验证集)
train_model(net, train_features, train_labels, None, None, best_config)

# 生成预测
net.eval()
with torch.no_grad():
    preds = net(test_features).numpy()

# 保存 CSV
SAVE_DIR = DATA_ROOT / "kaggle_house"
submission = pd.DataFrame({
    'Id': test_data["Id"],          # 第一列：房屋 ID
    'SalePrice': preds.reshape(-1)  # 第二列：预测出的房价。reshape(-1) 是为了确保变成 1 维数组
})

# index=False 不保存索引。
submission.to_csv(SAVE_DIR / "submission.csv", index=False)
print(f"预测完成！结果已保存至: {SAVE_DIR / "submission.csv"}")


>>> 开始评估[LINEAR] 模型...
折 1 | 训练 Log-RMSE: 0.1700 | 验证 Log-RMSE: 0.1571
折 2 | 训练 Log-RMSE: 0.1620 | 验证 Log-RMSE: 0.1895
折 3 | 训练 Log-RMSE: 0.1641 | 验证 Log-RMSE: 0.1686
折 4 | 训练 Log-RMSE: 0.1680 | 验证 Log-RMSE: 0.1544
折 5 | 训练 Log-RMSE: 0.1627 | 验证 Log-RMSE: 0.1828

平均验证 Log-RMSE: 0.1705

>>> 开始评估[MLP] 模型...
折 1 | 训练 Log-RMSE: 0.1050 | 验证 Log-RMSE: 0.1337
折 2 | 训练 Log-RMSE: 0.1129 | 验证 Log-RMSE: 0.1684
折 3 | 训练 Log-RMSE: 0.0969 | 验证 Log-RMSE: 0.1503
折 4 | 训练 Log-RMSE: 0.1048 | 验证 Log-RMSE: 0.1422
折 5 | 训练 Log-RMSE: 0.0952 | 验证 Log-RMSE: 0.1558

平均验证 Log-RMSE: 0.1501

实验结论: 
Linear 模型平均误差: 0.1705
MLP 模型平均误差:    0.1501

>>> 自动选择最优模型 [MLP] 生成提交文件...
预测完成！结果已保存至: /home/august/deepseek/pytorch_study/temp/data/kaggle_house/submission.csv
